# Kubernetes Metrics Data Exploration

This notebook explores the Kubernetes metrics data for failure prediction analysis. It's designed to be an interactive tool for understanding the relationship between various metrics and failure events.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Import custom functions
import sys
sys.path.append('..')
from data_generator import generate_kubernetes_data
from data_processor import preprocess_data

## Generate Sample Data

First, we'll generate a sample dataset of Kubernetes metrics with a specified failure rate.

In [ ]:
# Generate sample data
num_samples = 5000
failure_rate = 0.1  # 10% failure rate
time_steps = 30

data = generate_kubernetes_data(num_samples, failure_rate, time_steps)

# Display basic info
print(f"Dataset shape: {data.shape}")
print(f"Failure rate: {data['failure'].mean():.2%}")
data.head()

## Data Overview

Let's examine the data types and basic statistics of our metrics.

In [ ]:
# Display data info
data.info()

In [ ]:
# Display descriptive statistics
data.describe()

## Distribution of Metrics

Let's visualize the distribution of key metrics and see how they differ between failure and non-failure cases.

In [ ]:
# Function to plot distributions
def plot_metric_distribution(data, metric, title=None):
    plt.figure(figsize=(12, 6))
    
    # Plot for non-failure cases
    sns.histplot(data[data['failure'] == 0][metric], 
                 label='Normal', alpha=0.5, kde=True)
    
    # Plot for failure cases
    sns.histplot(data[data['failure'] == 1][metric], 
                 label='Failure', alpha=0.5, kde=True)
    
    plt.title(title or f'Distribution of {metric}')
    plt.legend()
    plt.show()

In [ ]:
# Plot distribution of CPU usage
plot_metric_distribution(data, 'cpu_usage_percent', 'CPU Usage Distribution')

In [ ]:
# Plot distribution of memory usage
plot_metric_distribution(data, 'memory_usage_percent', 'Memory Usage Distribution')

In [ ]:
# Plot distribution of disk usage
plot_metric_distribution(data, 'disk_usage_percent', 'Disk Usage Distribution')

## Correlation Analysis

Let's examine the correlation between different metrics and with failure events.

In [ ]:
# Calculate correlations
numeric_data = data.select_dtypes(include=['number'])
correlation_matrix = numeric_data.corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5, fmt='.2f')
plt.title('Correlation Matrix of Kubernetes Metrics')
plt.tight_layout()
plt.show()

## Time Series Analysis

Let's analyze how metrics change over time, particularly before failure events.

In [ ]:
# Convert timestamp to datetime if it's not already
if data['timestamp'].dtype != 'datetime64[ns]':
    data['timestamp'] = pd.to_datetime(data['timestamp'])

# Group by timestamp and calculate average metrics
time_data = data.groupby('timestamp').agg({
    'cpu_usage_percent': 'mean',
    'memory_usage_percent': 'mean',
    'disk_usage_percent': 'mean',
    'failure': 'mean'  # This gives us the proportion of failures at each timestamp
}).reset_index()

# Create a multi-line time series plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time_data['timestamp'],
    y=time_data['cpu_usage_percent'],
    name='CPU Usage %',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=time_data['timestamp'],
    y=time_data['memory_usage_percent'],
    name='Memory Usage %',
    line=dict(color='green')
))

fig.add_trace(go.Scatter(
    x=time_data['timestamp'],
    y=time_data['disk_usage_percent'],
    name='Disk Usage %',
    line=dict(color='orange')
))

# Create a secondary Y-axis for failure rate
fig.add_trace(go.Scatter(
    x=time_data['timestamp'],
    y=time_data['failure'] * 100,  # Convert to percentage
    name='Failure Rate %',
    line=dict(color='red', dash='dot'),
    yaxis='y2'
))

fig.update_layout(
    title='Time Series of Kubernetes Metrics and Failure Rates',
    xaxis_title='Time',
    yaxis_title='Usage Percentage',
    yaxis2=dict(
        title='Failure Rate %',
        overlaying='y',
        side='right'
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    height=600
)

fig.show()

## Failure Pattern Analysis

Let's examine specific patterns that precede failures to better understand what triggers them.

In [ ]:
# Get failure events
failure_events = data[data['failure'] == 1]
print(f"Total failure events: {len(failure_events)}")

# Count the number of failures by node
failure_by_node = failure_events['node'].value_counts()

# Plot failures by node
plt.figure(figsize=(10, 6))
failure_by_node.plot(kind='bar')
plt.title('Number of Failures by Node')
plt.xlabel('Node')
plt.ylabel('Failure Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze failure conditions
failure_conditions = {
    'CPU Exhaustion (>85%)': (failure_events['cpu_usage_percent'] > 85).sum(),
    'Memory Exhaustion (>90%)': (failure_events['memory_usage_percent'] > 90).sum(),
    'Disk Pressure (>85%)': (failure_events['disk_usage_percent'] > 85).sum(),
    'Memory Pressure': (failure_events['node_condition_memory_pressure'] == 1).sum(),
    'Disk Pressure': (failure_events['node_condition_disk_pressure'] == 1).sum(),
    'Network Unavailable': (failure_events['node_condition_network_unavailable'] == 1).sum(),
    'Node Not Ready': (failure_events['node_condition_ready'] == 0).sum(),
    'High Pod Restart (>=3)': (failure_events['pod_restart_count'] >= 3).sum(),
    'High Pending Pods (>=5)': (failure_events['pod_pending_count'] >= 5).sum()
}

# Convert to DataFrame for plotting
failure_patterns_df = pd.DataFrame(list(failure_conditions.items()), 
                                 columns=['Condition', 'Count'])
failure_patterns_df['Percentage'] = failure_patterns_df['Count'] / len(failure_events) * 100

# Sort by count
failure_patterns_df = failure_patterns_df.sort_values('Count', ascending=False)

# Plot
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='Percentage', y='Condition', data=failure_patterns_df, palette='viridis')
plt.title('Failure Pattern Distribution')
plt.xlabel('Percentage of Failures')
plt.ylabel('Condition')

# Add count labels
for i, row in enumerate(failure_patterns_df.itertuples()):
    ax.text(row.Percentage + 1, i, f'{row.Count} events', va='center')

plt.tight_layout()
plt.show()

## Preprocessing for Model Training

Now let's preprocess the data to prepare it for model training.

In [ ]:
# Preprocess data
preprocessed_data, preprocessing_meta = preprocess_data(data)
print(f"Shape after preprocessing: {preprocessed_data.shape}")
preprocessed_data.head()

## Save Processed Data

Save the processed data for model training.

In [ ]:
# Save to CSV
preprocessed_data.to_csv('../data/preprocessed_kubernetes_data.csv', index=False)
print("Data saved to ../data/preprocessed_kubernetes_data.csv")

## Conclusion

In this notebook, we've explored the Kubernetes metrics data and identified several patterns that are associated with failures. Key findings include:

1. CPU, memory, and disk usage distributions show clear differences between normal and failure cases.
2. Various node conditions like memory pressure and disk pressure are strong indicators of failures.
3. Pod restart counts and pending pod counts can also provide early warning signs.

These findings will guide our feature selection and model development in the subsequent notebooks.